In [0]:
# Silver Claims

from pyspark.sql import functions as F

CATALOG = "insurance_lakehouse"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
QUARANTINE_SCHEMA = "quarantine"

bronze_table = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_claims"
silver_table = f"{CATALOG}.{SILVER_SCHEMA}.silver_claims"
quarantine_table = f"{CATALOG}.{QUARANTINE_SCHEMA}.quarantine_invalid_claims"

valid_claim_status = ["open", "approved", "rejected", "under_review", "paid"]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{QUARANTINE_SCHEMA}")

claims_bronze = spark.table(bronze_table)

silver_policies = (
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_policies")
    .select(F.col("policy_id").alias("valid_policy_id"))
    .dropDuplicates()
)

silver_customers = (
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_customers")
    .select(F.col("customer_id").alias("valid_customer_id"))
    .dropDuplicates()
)

claims_prepared = (
    claims_bronze
    .withColumn("claim_id", F.trim(F.col("claim_id")))
    .withColumn("policy_id", F.trim(F.col("policy_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("claim_date", F.to_date(F.col("claim_date")))
    .withColumn("claim_type", F.lower(F.trim(F.col("claim_type"))))
    .withColumn("claim_status", F.lower(F.trim(F.col("claim_status"))))
    .withColumn("reported_channel", F.lower(F.trim(F.col("reported_channel"))))
    .withColumn("claim_amount", F.col("claim_amount").cast("double"))
    .withColumn("fraud_flag", F.col("fraud_flag").cast("boolean"))
    .withColumn("created_at", F.to_timestamp(F.col("created_at")))
)

claims_joined = (
    claims_prepared
    .join(silver_policies, F.col("policy_id") == F.col("valid_policy_id"), "left")
    .join(silver_customers, F.col("customer_id") == F.col("valid_customer_id"), "left")
    .withColumn("policy_exists", F.col("valid_policy_id").isNotNull())
    .withColumn("customer_exists", F.col("valid_customer_id").isNotNull())
    .drop("valid_policy_id", "valid_customer_id")
)

duplicate_claim_ids = (
    claims_joined
    .groupBy("claim_id")
    .count()
    .filter(F.col("claim_id").isNotNull() & (F.col("count") > 1))
    .select("claim_id")
    .withColumn("is_duplicate_claim_id", F.lit(True))
)

claims_checked = (
    claims_joined
    .join(duplicate_claim_ids, on="claim_id", how="left")
    .withColumn(
        "is_duplicate_claim_id",
        F.coalesce(F.col("is_duplicate_claim_id"), F.lit(False))
    )
)

invalid_condition = (
    F.col("claim_id").isNull()
    | F.col("is_duplicate_claim_id")
    | F.col("policy_id").isNull()
    | F.col("customer_id").isNull()
    | (~F.col("policy_exists"))
    | (~F.col("customer_exists"))
    | F.col("claim_amount").isNull()
    | (F.col("claim_amount") <= 0)
    | F.col("claim_status").isNull()
    | (~F.col("claim_status").isin(valid_claim_status))
)

error_reason = (
    F.when(F.col("claim_id").isNull(), F.lit("missing_claim_id"))
    .when(F.col("is_duplicate_claim_id"), F.lit("duplicate_claim_id"))
    .when(F.col("policy_id").isNull(), F.lit("missing_policy_id"))
    .when(~F.col("policy_exists"), F.lit("missing_policy_reference"))
    .when(F.col("customer_id").isNull(), F.lit("missing_customer_id"))
    .when(~F.col("customer_exists"), F.lit("missing_customer_reference"))
    .when(F.col("claim_amount").isNull(), F.lit("missing_claim_amount"))
    .when(F.col("claim_amount") <= 0, F.lit("invalid_claim_amount"))
    .when(F.col("claim_status").isNull(), F.lit("missing_claim_status"))
    .when(~F.col("claim_status").isin(valid_claim_status), F.lit("invalid_claim_status"))
    .otherwise(F.lit("unknown_claim_quality_issue"))
)

invalid_claims = (
    claims_checked
    .filter(invalid_condition)
    .withColumn("record_id", F.col("claim_id").cast("string"))
    .withColumn("source_table", F.lit("bronze_claims"))
    .withColumn("error_reason", error_reason)
    .withColumn("error_severity", F.lit("HIGH"))
    .withColumn("quarantine_timestamp", F.current_timestamp())
    .withColumn(
        "original_record_json",
        F.to_json(F.struct(*[F.col(c) for c in claims_prepared.columns]))
    )
    .select(
        "record_id",
        "source_table",
        "error_reason",
        "error_severity",
        "quarantine_timestamp",
        "source_file_name",
        "ingest_run_id",
        "original_record_json"
    )
)

valid_claims = (
    claims_checked
    .filter(~invalid_condition)
    .drop("policy_exists", "customer_exists", "is_duplicate_claim_id")
    .dropDuplicates(["claim_id"])
)

valid_claims.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table)

invalid_claims.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(quarantine_table)

print("Bronze claims:", claims_bronze.count())
print("Silver claims:", spark.table(silver_table).count())
print("Quarantine claims:", spark.table(quarantine_table).count())